# КТ №5. Allure

In [ ]:
%pip install -q pytest requests "pydantic>=2,<3" allure-pytest


In [ ]:
%%writefile test_petstore.py
import json
from uuid import uuid4
from typing import Literal

import allure
import pytest
import requests
from pydantic import BaseModel, ConfigDict, Field, TypeAdapter


class User(BaseModel):
    model_config = ConfigDict(strict=True)
    id: int
    username: str
    firstName: str = "Emil"
    lastName: str = "Gainulin"
    email: str = "emil@example.com"
    password: str = "test123"
    phone: str = "1234567890"
    userStatus: int = 0


class Pet(BaseModel):
    model_config = ConfigDict(strict=True)
    id: int
    name: str
    photoUrls: list[str] = Field(default_factory=list)
    status: Literal["available", "pending", "sold"] = "available"


class Order(BaseModel):
    model_config = ConfigDict(strict=True)
    id: int
    petId: int
    quantity: int = Field(default=1, gt=0)
    status: Literal["placed", "approved", "delivered"] = "placed"
    complete: bool = False


class ApiResponse(BaseModel):
    model_config = ConfigDict(strict=True)
    code: int
    type: str
    message: str


class BaseRequest:
    def request(self, method, endpoint, data=None):
        with allure.step(f"{method} {endpoint}"):
            if data is not None:
                allure.attach(json.dumps(data, ensure_ascii=False, indent=2),
                              "Запрос", allure.attachment_type.JSON)
            response = requests.request(
                method, "https://petstore.swagger.io/v2" + endpoint,
                json=data, headers={"Accept": "application/json"}, timeout=30
            )
            allure.attach(str(response.status_code), "Код ответа",
                          allure.attachment_type.TEXT)
            allure.attach(response.text, "Ответ", allure.attachment_type.TEXT)
            return response


@pytest.fixture
def api():
    return BaseRequest()


@pytest.fixture
def user_data():
    number = uuid4().int % 1000000000
    return User(id=number, username="kt5_" + uuid4().hex)


@pytest.fixture
def saved_user(api, user_data):
    response = api.request("POST", "/user", user_data.model_dump(mode="json"))
    assert response.status_code == 200
    yield user_data
    api.request("DELETE", "/user/" + user_data.username)


@pytest.fixture
def order_data():
    return Order(id=uuid4().int % 1000000000 + 1000, petId=1)


@pytest.fixture
def saved_order(api, order_data):
    response = api.request("POST", "/store/order", order_data.model_dump(mode="json"))
    assert response.status_code == 200
    yield order_data
    api.request("DELETE", f"/store/order/{order_data.id}")


@allure.feature("User")
@allure.title("Создание пользователя")
def test_create_user(api, user_data):
    try:
        response = api.request("POST", "/user", user_data.model_dump(mode="json"))
        assert response.status_code == 200
        result = ApiResponse.model_validate(response.json())
        assert result.code == 200
        assert result.message == str(user_data.id)
    finally:
        api.request("DELETE", "/user/" + user_data.username)


@allure.feature("User")
@allure.title("Получение пользователя")
def test_get_user(api, saved_user):
    response = api.request("GET", "/user/" + saved_user.username)
    assert response.status_code == 200
    result = User.model_validate(response.json())
    assert result.id == saved_user.id
    assert result.username == saved_user.username


@allure.feature("User")
@allure.title("Изменение пользователя")
def test_update_user(api, saved_user):
    saved_user.firstName = "Artur"
    response = api.request("PUT", "/user/" + saved_user.username,
                           saved_user.model_dump(mode="json"))
    assert response.status_code == 200
    assert ApiResponse.model_validate(response.json()).code == 200
    response = api.request("GET", "/user/" + saved_user.username)
    assert response.status_code == 200
    assert User.model_validate(response.json()).firstName == "Artur"


@allure.feature("User")
@allure.title("Удаление пользователя")
def test_delete_user(api, saved_user):
    response = api.request("DELETE", "/user/" + saved_user.username)
    assert response.status_code == 200
    assert ApiResponse.model_validate(response.json()).code == 200
    response = api.request("GET", "/user/" + saved_user.username)
    assert response.status_code == 404


@allure.feature("Store")
@allure.title("Создание заказа")
def test_create_order(api, order_data):
    try:
        response = api.request("POST", "/store/order", order_data.model_dump(mode="json"))
        assert response.status_code == 200
        result = Order.model_validate(response.json())
        assert result == order_data
    finally:
        api.request("DELETE", f"/store/order/{order_data.id}")


@allure.feature("Store")
@allure.title("Получение заказа")
def test_get_order(api, saved_order):
    response = api.request("GET", f"/store/order/{saved_order.id}")
    assert response.status_code == 200
    result = Order.model_validate(response.json())
    assert result == saved_order


@allure.feature("Store")
@allure.title("Удаление заказа")
def test_delete_order(api, saved_order):
    response = api.request("DELETE", f"/store/order/{saved_order.id}")
    assert response.status_code == 200
    assert ApiResponse.model_validate(response.json()).code == 200
    response = api.request("GET", f"/store/order/{saved_order.id}")
    assert response.status_code == 404


@allure.feature("Store")
@allure.title("Получение количества питомцев")
def test_inventory(api):
    response = api.request("GET", "/store/inventory")
    assert response.status_code == 200
    result = TypeAdapter(dict[str, int]).validate_python(response.json(), strict=True)
    assert all(count >= 0 for count in result.values())


@pytest.fixture
def pet_data():
    return Pet(id=uuid4().int % 1000000000 + 1000, name="Barsik")


@pytest.fixture
def saved_pet(api, pet_data):
    response = api.request("POST", "/pet", pet_data.model_dump(mode="json"))
    assert response.status_code == 200
    yield pet_data
    api.request("DELETE", f"/pet/{pet_data.id}")


@allure.feature("Pet")
@allure.title("Создание питомца")
def test_create_pet(api, pet_data):
    try:
        response = api.request("POST", "/pet", pet_data.model_dump(mode="json"))
        assert response.status_code == 200
        result = Pet.model_validate(response.json())
        assert result == pet_data
    finally:
        api.request("DELETE", f"/pet/{pet_data.id}")


@allure.feature("Pet")
@allure.title("Получение питомца")
def test_get_pet(api, saved_pet):
    response = api.request("GET", f"/pet/{saved_pet.id}")
    assert response.status_code == 200
    result = Pet.model_validate(response.json())
    assert result == saved_pet


@allure.feature("Pet")
@allure.title("Изменение питомца")
def test_update_pet(api, saved_pet):
    saved_pet.name = "Murzik"
    response = api.request("PUT", "/pet", saved_pet.model_dump(mode="json"))
    assert response.status_code == 200
    assert Pet.model_validate(response.json()) == saved_pet
    response = api.request("GET", f"/pet/{saved_pet.id}")
    assert response.status_code == 200
    assert Pet.model_validate(response.json()) == saved_pet


@allure.feature("Pet")
@allure.title("Удаление питомца")
def test_delete_pet(api, saved_pet):
    response = api.request("DELETE", f"/pet/{saved_pet.id}")
    assert response.status_code == 200
    assert ApiResponse.model_validate(response.json()).code == 200
    response = api.request("GET", f"/pet/{saved_pet.id}")
    assert response.status_code == 404


In [ ]:
!python -m pytest test_petstore.py -v --alluredir=allure-results --clean-alluredir


In [ ]:
!npx --yes allure@3 awesome ./allure-results --single-file --output ./allure-report


In [ ]:
from pathlib import Path
from google.colab import files

reports = list(Path("allure-report").rglob("*.html"))
if len(reports) != 1:
    raise RuntimeError("HTML-отчёт не найден. Проверь предыдущую ячейку.")
files.download(str(reports[0]))
